# `SmoothedSurface`

`nematics3d.geometry.SmoothedSurface` smooths a discrete surface with a wavelength-based Taubin filter while strongly suppressing the systematic shrinkage associated with ordinary Laplacian smoothing.

The main user-facing smoothing scale is `cutoff_wavelength`. The iteration count and the dimensional Taubin coefficients are resolved automatically from the requested wavelength and the spectrum of the input mesh.


## What `SmoothedSurface` is for

Use `SmoothedSurface` when a surface mesh contains short-wavelength geometric roughness that should be removed without deliberately shrinking the whole closed surface.

The input may be a closed surface. Internally, Nematics3D converts the geometry to a triangle `pyvista.PolyData`, constructs a cotangent Laplace--Beltrami operator on the initial mesh, and keeps that operator fixed during the smoothing pass. It then chooses the smallest stable number of Taubin iteration pairs and computes the corresponding `lambda` and `mu` automatically.

This class smooths the **geometry** of the surface. Sampling or smoothing scalar/vector functions defined on the surface is a separate problem and is not covered here.


## Setup

**For readers who are only interested in the tutorial, this section can be safely skipped.** The next cell only imports NumPy, PyVista, and Nematics3D through its public package interface.


In [ ]:
import numpy as np
import pyvista as pv
import nematics3d as n3d


## Build a noisy closed surface

For a self-contained example, start from a triangulated sphere and add deterministic radial roughness. This is only demonstration data; in normal use, `surface` would be the mesh produced by your own reconstruction or analysis pipeline.


In [ ]:
surface = pv.Sphere(
    radius=10.0,
    theta_resolution=80,
    phi_resolution=80,
).triangulate()

points = np.asarray(surface.points, dtype=float)
radius = np.linalg.norm(points, axis=1)
direction = points / radius[:, None]

x, y, z = direction.T
roughness = 0.25 * np.sin(18.0 * np.arctan2(y, x)) * (1.0 - z**2)

surface_noisy = surface.copy()
surface_noisy.points = points + roughness[:, None] * direction


## Minimal example

The shortest useful call needs the input surface and `cutoff_wavelength`:

```python
smoothed = n3d.SmoothedSurface(
    surface_noisy,
    cutoff_wavelength=2.0,
)
```

`cutoff_wavelength` has the **same physical length unit as the surface coordinates**. It is not a mesh spacing, displacement distance, or number of vertices.

Nematics3D defines it as the `-3 dB` amplitude cutoff of the **complete** smoothing pass. If

$$
\kappa_c = \left(\frac{2\pi}{\ell_c}\right)^2,
$$

where $\ell_c$ is `cutoff_wavelength`, then the complete filter satisfies

$$
G_N(\kappa_c)=\frac{1}{\sqrt{2}}.
$$

Thus a surface mode with wavelength exactly equal to `cutoff_wavelength` retains $1/\sqrt{2}$ of its original amplitude. Shorter-wavelength structure is suppressed more strongly, while longer-wavelength structure is preserved more strongly.


In [ ]:
smoothed = n3d.SmoothedSurface(
    surface_noisy,
    cutoff_wavelength=2.0,
)

smoothed.result


## Access the result

`smoothed.result` is the final geometry-only `pyvista.PolyData`. It preserves the triangle connectivity of the triangulated initial surface and replaces its points by the smoothed coordinates.

If only the coordinates are needed, use `smoothed.vertices`. This is a read-only NumPy array with shape `(n_points, 3)`.


In [ ]:
surface_smooth = smoothed.result
vertices_smooth = smoothed.vertices

print(type(surface_smooth))
print(vertices_smooth.shape)
print(smoothed.calc_status)


## Compare the original and smoothed surfaces

Because both objects are PyVista surfaces, they can be inspected with ordinary PyVista tools. For example, the following cell displays the noisy and smoothed surfaces side by side.


In [ ]:
plotter = pv.Plotter(shape=(1, 2))

plotter.subplot(0, 0)
plotter.add_text("Input")
plotter.add_mesh(surface_noisy, smooth_shading=True)

plotter.subplot(0, 1)
plotter.add_text("Smoothed")
plotter.add_mesh(surface_smooth, smooth_shading=True)

plotter.link_views()
plotter.show()


## Smoothing options

`SmoothedSurface` currently has two smoothing options:

- `cutoff_wavelength`: the required physical `-3 dB` cutoff wavelength of the complete smoothing pass.
- `taubin_ratio`: the dimensionless ratio $r=-\mu/\lambda$.

Normally, only `cutoff_wavelength` needs to be specified. `taubin_ratio` uses the library default `1.0674`.

The raw Taubin coefficients and iteration count are deliberately **not** user options. They depend on the physical cutoff and on the discrete spectrum of the current mesh, so `SmoothedSurface` resolves them internally.


In [ ]:
smoothed.opts


### Passing an `OptsSmoothedSurface` object

As with other `HostBase` objects, options can also be constructed separately and passed through `opts`.


In [ ]:
opts = n3d.OptsSmoothedSurface(
    cutoff_wavelength=2.0,
)

smoothed_from_opts = n3d.SmoothedSurface(
    surface_noisy,
    opts=opts,
)


### Overriding the library default

If there is a specific reason to change the Taubin asymmetry, `taubin_ratio` can be passed explicitly:


In [ ]:
smoothed_custom_ratio = n3d.SmoothedSurface(
    surface_noisy,
    cutoff_wavelength=2.0,
    taubin_ratio=1.08,
)


## Change the smoothing scale after construction

The smoothing options remain editable through the normal `HostBase` commit mechanism. For example:


In [ ]:
smoothed.act_commit(
    cutoff_wavelength=2.5,
)


When only smoothing options such as `cutoff_wavelength` or `taubin_ratio` change, Nematics3D reuses the Laplace--Beltrami operator constructed from the current raw surface. It recomputes the Taubin filter parameters and the smoothed result, but does not rebuild the mesh operator unnecessarily.

This is also important mathematically: during each smoothing pass, the operator is fixed to the initial geometry rather than recomputed after every vertex update. Therefore the smoothing operation remains a literal linear spectral filter for that mesh.


## Inspect the automatically resolved filter

Most users do not need these quantities, but they are exposed for diagnostics and reproducibility:


In [ ]:
print("cutoff kappa :", smoothed.calc_kappa_cutoff)
print("maximum kappa:", smoothed.calc_kappa_max)
print("Taubin pairs :", smoothed.calc_iterations)
print("lambda       :", smoothed.calc_lambda)
print("mu           :", smoothed.calc_mu)


The discrete operator uses the convention

$$
L\phi=-\kappa\phi,\qquad \kappa\ge 0.
$$

For one Taubin pair, the modal amplitude is multiplied by

$$
g(\kappa)=(1-\lambda\kappa)(1-\mu\kappa).
$$

After $N$ pairs,

$$
G_N(\kappa)=g(\kappa)^N.
$$

For the current mesh and requested cutoff, Nematics3D chooses the smallest positive $N$ for which the highest resolved mesh frequency is not amplified in magnitude. It then solves for `lambda` and `mu` so that the complete filter has the requested `-3 dB` cutoff.


### Verify the cutoff definition

The following diagnostic evaluates the complete transfer function at the resolved cutoff eigenvalue. It should be numerically equal to $1/\sqrt{2}$.


In [ ]:
kappa_c = smoothed.calc_kappa_cutoff
lambda_ = smoothed.calc_lambda
mu = smoothed.calc_mu
N = smoothed.calc_iterations

gain_at_cutoff = (
    (1.0 - lambda_ * kappa_c)
    * (1.0 - mu * kappa_c)
) ** N

print(gain_at_cutoff)
print(1.0 / np.sqrt(2.0))


## Choosing `cutoff_wavelength`

The parameter should be chosen from the **physical geometric scale that you want to remove**, not from the desired number of iterations.

A useful interpretation is:

- structures with wavelength much longer than `cutoff_wavelength` are largely preserved;
- a mode at exactly `cutoff_wavelength` is reduced to $1/\sqrt{2}$ amplitude;
- structures substantially shorter than `cutoff_wavelength` are suppressed more strongly.

For example, if the physically meaningful surface shape varies on scales of roughly `10` length units but the unwanted roughness is concentrated around `1`--`2` length units, a cutoff of order `2` is a reasonable quantity to test.

There is no universal default because the correct value depends on the physical length scale of the surface.


## What "without shrinkage" means here

Taubin smoothing is used because ordinary positive Laplacian smoothing tends to move a curved closed surface inward systematically. The positive/negative Taubin pair compensates much of this low-frequency shrinkage.

This should not be interpreted as an exact volume constraint. `SmoothedSurface` does **not** enforce constant enclosed volume or constant area. If exact volume preservation is required, that is a different constraint and should be handled explicitly rather than inferred from the Taubin filter.


## Mesh requirements and failure cases

`SmoothedSurface` requires a valid surface that can be converted to a non-empty triangle mesh. In particular:

- triangle coordinates must be finite;
- triangles must not be degenerate or numerically singular;
- every retained vertex must belong to valid surface geometry and receive positive lumped mass;
- the requested cutoff and `taubin_ratio` must admit a finite stable Taubin filter for the discrete spectrum of the mesh.

If these requirements are violated, construction or recommitting the smoothing configuration raises `SurfaceSmoothingConfigError` rather than silently returning an unstable result.

Because `cutoff_wavelength` is a physical length, changing the mesh resolution while representing the same physical surface should not require redefining the meaning of the parameter. However, the discrete spectrum and therefore the automatically selected iteration count can change with the mesh.


## Summary

For normal use, the intended workflow is simply:

```python
smoothed = n3d.SmoothedSurface(
    surface,
    cutoff_wavelength=desired_physical_scale,
)

surface_smooth = smoothed.result
```

`cutoff_wavelength` is the main physical control. `taubin_ratio` normally stays at its library default, while `iterations`, `lambda`, and `mu` are implementation-resolved quantities that remain available for inspection.

Surface-function sampling and interpolation are intentionally outside the scope of this tutorial.
